# S30 - Evidencia de aprendizaje 1
## Análisis y herramientas de extracción de datos

**Nota:** Para esta práctica usé Books to Scrape, una tienda de ejemplo creada para practicar web scraping. Los precios y las calificaciones son datos de prueba, no precios reales del mercado.

### Producto seleccionado
**Producto:** *A Light in the Attic*
**Sitio:** Books to Scrape
**URL:** https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html

###CRISP-DM
1. **Comprensión del negocio:** definir qué información del producto se necesita.
2. **Comprensión de los datos:** revisar qué datos aparecen en la página.
3. **Preparación de los datos:** limpiar los espacios y organizar los campos.
4. **Modelado:** hacer la extracción con BeautifulSoup, Selenium y Scrapy.
5. **Evaluación:** comparar los datos y los tiempos obtenidos.



In [8]:
# Instalo las bibliotecas que voy a usar.
# Selenium necesita un navegador dentro del entorno de Colab.
import subprocess
import sys

paquetes = [
    "requests",
    "beautifulsoup4",
    "lxml",
    "selenium",
    "google-colab-selenium",
    "scrapy",
    "pandas",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *paquetes],
    check=True,
)
print("Bibliotecas instaladas correctamente.")
print("Chrome/ChromeDriver se prepararán dentro de Colab al ejecutar la celda de Selenium.")


Bibliotecas instaladas correctamente.
Chrome/ChromeDriver se prepararán dentro de Colab al ejecutar la celda de Selenium.


## 2. Configuración general

Para comparar los metodos, se usa la misma pagina y los mismos 10 campos. Tambien se registra el tiempo y se guardan los errores por si alguno de los métodos no termina.

In [9]:
from pathlib import Path
from time import perf_counter
import json
import subprocess
import sys
import pandas as pd

PRODUCT_URL = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; AcademicWebScrapingPractice/1.0; educational use)"
}

# Estos son los mismos campos para los tres métodos.
CAMPOS = [
    "titulo",
    "precio",
    "disponibilidad",
    "upc",
    "tipo_producto",
    "precio_sin_impuestos",
    "precio_con_impuestos",
    "impuesto",
    "numero_resenas",
    "calificacion",
]

# Se relacionan los nombres de la página con los nombres que usamos en el código.
MAPA_CAMPOS = {
    "UPC": "upc",
    "Product Type": "tipo_producto",
    "Price (excl. tax)": "precio_sin_impuestos",
    "Price (incl. tax)": "precio_con_impuestos",
    "Tax": "impuesto",
    "Availability": "disponibilidad",
    "Number of reviews": "numero_resenas",
}

def rating_a_texto(clases):
    """Pasa la calificación a un texto como "3 de 5"."""
    if isinstance(clases, str):
        palabras = clases.split()
    else:
        palabras = list(clases or [])
    equivalencias = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    for palabra in palabras:
        if palabra in equivalencias:
            return f"{equivalencias[palabra]} de 5"
    return ""

def normalizar_registro(registro):
    """Deja los mismos campos en los tres resultados."""
    limpio = {}
    for campo in CAMPOS:
        valor = registro.get(campo, "")
        texto = " ".join(str(valor).split()) if valor is not None else ""
        # Corrige el símbolo de libra si aparece mal escrito.
        texto = texto.replace("Â£", "£")
        limpio[campo] = texto
    return limpio

print("URL del producto:", PRODUCT_URL)
print("Campos que compararemos:", len(CAMPOS))

URL del producto: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
Campos que compararemos: 10


## 3. Método 1: BeautifulSoup

BeautifulSoup sirve para buscar información dentro del HTML de una página. Primero se descarga el contenido y luego se buscan el título, el precio, la disponibilidad y los demás campos que se necesita.

In [10]:
import requests
from bs4 import BeautifulSoup

resultado_bs = None
error_bs = None
tiempo_bs = None

try:
    inicio = perf_counter()
    respuesta = requests.get(PRODUCT_URL, headers=HEADERS, timeout=25)
    respuesta.raise_for_status()

    # Se usa el contenido original para que se lea bien el símbolo £.
    soup = BeautifulSoup(respuesta.content, "lxml")

    registro = {campo: "" for campo in CAMPOS}
    titulo = soup.select_one("div.product_main h1") or soup.select_one("h1")
    precio = soup.select_one("div.product_main p.price_color") or soup.select_one("p.price_color")
    disponibilidad = soup.select_one("div.product_main p.instock.availability") or soup.select_one("p.availability")
    rating = soup.select_one("div.product_main p.star-rating") or soup.select_one("p.star-rating")

    registro["titulo"] = titulo.get_text(" ", strip=True) if titulo else ""
    registro["precio"] = precio.get_text(" ", strip=True) if precio else ""
    registro["disponibilidad"] = disponibilidad.get_text(" ", strip=True) if disponibilidad else ""
    registro["calificacion"] = rating_a_texto(rating.get("class", [])) if rating else ""

    for fila in soup.select("table.table-striped tr, table tr"):
        th = fila.select_one("th")
        td = fila.select_one("td")
        if th and td:
            etiqueta = th.get_text(" ", strip=True)
            valor = td.get_text(" ", strip=True)
            campo = MAPA_CAMPOS.get(etiqueta)
            if campo:
                registro[campo] = valor

    resultado_bs = normalizar_registro(registro)
    tiempo_bs = perf_counter() - inicio
    print("BeautifulSoup terminó correctamente.")
    display(pd.DataFrame([resultado_bs]))
    print(f"Tiempo de ejecución: {tiempo_bs:.3f} segundos")
except Exception as exc:
    error_bs = f"{type(exc).__name__}: {exc}"
    print("BeautifulSoup no pudo completar la extracción:", error_bs)
    print("Revisa la conexión a internet o vuelve a ejecutar esta celda más tarde.")

BeautifulSoup terminó correctamente.


,titulo,precio,disponibilidad,upc,tipo_producto,precio_sin_impuestos,precio_con_impuestos,impuesto,numero_resenas,calificacion
0,A Light in the Attic,£51.77,In stock (22 available),a897fe39b1053632,Books,£51.77,£51.77,£0.00,0,3 de 5


Tiempo de ejecución: 0.882 segundos


## 4. Método 2: Selenium

Selenium permite abrir y manejar un navegador mediante código. En esta práctica lo usamos para abrir la página del libro y leer los mismos campos que buscamos con BeautifulSoup.


In [11]:
import os
from pathlib import Path
from time import perf_counter
import selenium
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

resultado_selenium = None
error_selenium = None
tiempo_selenium = None
driver = None

try:
    inicio = perf_counter()
    opciones = Options()
    # Esta función prepara el navegador dentro de Colab.
    opciones.add_argument("--window-size=1365,1000")
    opciones.add_argument("--disable-gpu")

    print("Versión de Selenium:", selenium.__version__)
    print("Preparando el navegador de Selenium dentro de Google Colab...")
    driver = gs.Chrome(options=opciones)
    driver.set_page_load_timeout(40)
    driver.get(PRODUCT_URL)

    WebDriverWait(driver, 25).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.product_main h1"))
    )

    registro = {campo: "" for campo in CAMPOS}
    registro["titulo"] = driver.find_element(By.CSS_SELECTOR, "div.product_main h1").text.strip()
    registro["precio"] = driver.find_element(By.CSS_SELECTOR, "div.product_main p.price_color").text.strip()

    try:
        elemento_disp = driver.find_element(By.CSS_SELECTOR, "div.product_main p.instock.availability")
    except Exception:
        elemento_disp = driver.find_element(By.CSS_SELECTOR, "div.product_main p.availability")
    registro["disponibilidad"] = elemento_disp.text.strip()

    try:
        elemento_rating = driver.find_element(By.CSS_SELECTOR, "div.product_main p.star-rating")
        registro["calificacion"] = rating_a_texto(elemento_rating.get_attribute("class"))
    except Exception:
        registro["calificacion"] = ""

    for fila in driver.find_elements(By.CSS_SELECTOR, "table.table-striped tr"):
        try:
            etiqueta = fila.find_element(By.CSS_SELECTOR, "th").text.strip()
            valor = fila.find_element(By.CSS_SELECTOR, "td").text.strip()
            campo = MAPA_CAMPOS.get(etiqueta)
            if campo:
                registro[campo] = valor
        except Exception:
            # Si una fila no tiene los datos esperados, continua con la siguiente.
            continue

    resultado_selenium = normalizar_registro(registro)
    tiempo_selenium = perf_counter() - inicio
    print("Selenium terminó correctamente.")
    display(pd.DataFrame([resultado_selenium]))
    print(f"Tiempo de ejecución: {tiempo_selenium:.3f} segundos")

except Exception as exc:
    error_selenium = f"{type(exc).__name__}: {exc}"
    print("Selenium no pudo completar la extracción:", error_selenium)
    print("Si la preparación del navegador falló, confirma que Colab tenga acceso a Internet y vuelve a ejecutar esta celda.")
finally:
    if driver is not None:
        try:
            driver.quit()
        except Exception:
            pass


Versión de Selenium: 4.51.0
Preparando el navegador de Selenium dentro de Google Colab...


<IPython.core.display.Javascript object>

Selenium terminó correctamente.


,titulo,precio,disponibilidad,upc,tipo_producto,precio_sin_impuestos,precio_con_impuestos,impuesto,numero_resenas,calificacion
0,A Light in the Attic,£51.77,In stock (22 available),a897fe39b1053632,Books,£51.77,£51.77,£0.00,0,3 de 5


Tiempo de ejecución: 6.679 segundos


## 5. Método 3: Scrapy

Scrapy permite organizar la extracción con un *spider*. El spider abre la página, busca los datos con selectores CSS y guarda la información en un registro.


In [12]:
resultado_scrapy = None
error_scrapy = None
tiempo_scrapy = None

spider_code = r"""
import scrapy

class ProductoSpider(scrapy.Spider):
    name = "producto_scraping_ea1"
    start_urls = ["https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"]
    custom_settings = {
        "ROBOTSTXT_OBEY": True,
        "DOWNLOAD_DELAY": 1,
        "CONCURRENT_REQUESTS": 1,
        "LOG_LEVEL": "ERROR",
        "USER_AGENT": "AcademicWebScrapingPractice/1.0 (educational use)",
    }

    def parse(self, response):
        registro = {
            "titulo": response.css("div.product_main h1::text").get(default="").strip(),
            "precio": response.css("div.product_main p.price_color::text").get(default="").strip(),
            "disponibilidad": " ".join(response.css("div.product_main p.instock.availability ::text").getall()).strip(),
            "upc": "",
            "tipo_producto": "",
            "precio_sin_impuestos": "",
            "precio_con_impuestos": "",
            "impuesto": "",
            "numero_resenas": "",
            "calificacion": "",
        }

        rating_clases = response.css("div.product_main p.star-rating::attr(class)").get(default="")
        palabras = rating_clases.split()
        equivalencias = {"One": "1 de 5", "Two": "2 de 5", "Three": "3 de 5", "Four": "4 de 5", "Five": "5 de 5"}
        registro["calificacion"] = next((equivalencias[p] for p in palabras if p in equivalencias), "")

        mapa = {
            "UPC": "upc",
            "Product Type": "tipo_producto",
            "Price (excl. tax)": "precio_sin_impuestos",
            "Price (incl. tax)": "precio_con_impuestos",
            "Tax": "impuesto",
            "Availability": "disponibilidad",
            "Number of reviews": "numero_resenas",
        }
        for fila in response.css("table.table-striped tr"):
            etiqueta = fila.css("th::text").get(default="").strip()
            valor = fila.css("td::text").get(default="").strip()
            campo = mapa.get(etiqueta)
            if campo and valor:
                registro[campo] = valor

        # Si la tabla trae disponibilidad, se usa ese valor. Si no, queda el de product_main.
        yield registro
"""

try:
    inicio = perf_counter()
    carpeta = Path.cwd()
    archivo_spider = carpeta / "producto_spider_ea1.py"
    archivo_salida = carpeta / "resultado_scrapy_ea1.json"
    archivo_spider.write_text(spider_code, encoding="utf-8")
    if archivo_salida.exists():
        archivo_salida.unlink()

    proceso = subprocess.run(
        [sys.executable, "-m", "scrapy", "runspider", str(archivo_spider), "-O", str(archivo_salida)],
        capture_output=True,
        text=True,
        timeout=90,
        cwd=str(carpeta),
    )
    if proceso.returncode != 0:
        raise RuntimeError((proceso.stderr or proceso.stdout or "Scrapy terminó con un error.")[-2500:])
    if not archivo_salida.exists() or archivo_salida.stat().st_size == 0:
        raise RuntimeError("Scrapy no generó el archivo JSON esperado.")

    with archivo_salida.open("r", encoding="utf-8") as archivo:
        datos_scrapy = json.load(archivo)
    if not datos_scrapy:
        raise RuntimeError("Scrapy no devolvió registros. Revisa el acceso al sitio y los selectores.")

    resultado_scrapy = normalizar_registro(datos_scrapy[0])
    tiempo_scrapy = perf_counter() - inicio
    print("Scrapy terminó correctamente.")
    display(pd.DataFrame([resultado_scrapy]))
    print(f"Tiempo de ejecución: {tiempo_scrapy:.3f} segundos")
except Exception as exc:
    error_scrapy = f"{type(exc).__name__}: {exc}"
    print("Scrapy no pudo completar la extracción:", error_scrapy)
    print("Revisa la salida de error. El notebook no detiene las comparaciones de los otros métodos.")

Scrapy terminó correctamente.


,titulo,precio,disponibilidad,upc,tipo_producto,precio_sin_impuestos,precio_con_impuestos,impuesto,numero_resenas,calificacion
0,A Light in the Attic,£51.77,In stock (22 available),a897fe39b1053632,Books,£51.77,£51.77,£0.00,0,3 de 5


Tiempo de ejecución: 4.666 segundos


## 6. Comparación de resultados y métricas

Aquí se comparan los datos que devolvió cada método.
- **Campos completos:** cuántos de los 10 campos tienen un valor.
- **Completitud (%):** el porcentaje de campos obtenidos.
- **Tiempo (s):** cuánto tardó cada método en esta ejecución.
- **Coincidencia:** si los métodos obtuvieron el mismo valor para un campo.
Los tiempos pueden cambiar entre ejecuciones.

In [13]:
resultados_por_metodo = {
    "BeautifulSoup": (resultado_bs, tiempo_bs, error_bs),
    "Selenium": (resultado_selenium, tiempo_selenium, error_selenium),
    "Scrapy": (resultado_scrapy, tiempo_scrapy, error_scrapy),
}

resultados_validos = {
    nombre: registro
    for nombre, (registro, _, _) in resultados_por_metodo.items()
    if registro is not None
}

if resultados_validos:
    comparacion = pd.DataFrame(resultados_validos).T
    comparacion.index.name = "Metodo"
    print("Comparación de los campos extraídos:")
    display(comparacion)

    metricas = []
    for nombre, registro in resultados_validos.items():
        valores = [registro.get(campo, "") for campo in CAMPOS]
        completos = sum(bool(str(valor).strip()) for valor in valores)
        _, tiempo, _ = resultados_por_metodo[nombre]
        metricas.append({
            "Método": nombre,
            "Campos completos": completos,
            "Campos esperados": len(CAMPOS),
            "Completitud (%)": round(completos / len(CAMPOS) * 100, 2),
            "Tiempo (s)": round(tiempo, 3) if tiempo is not None else None,
        })
    df_metricas = pd.DataFrame(metricas)
    print("Métricas por método:")
    display(df_metricas)

    comparaciones_campo = []
    for campo in CAMPOS:
        pares = [(metodo, str(reg.get(campo, "")).strip()) for metodo, reg in resultados_validos.items()]
        no_vacios = [(metodo, valor) for metodo, valor in pares if valor]
        if len(no_vacios) >= 2:
            valores_normalizados = {" ".join(valor.split()).casefold() for _, valor in no_vacios}
            comparaciones_campo.append({
                "Campo": campo,
                "Métodos con valor": len(no_vacios),
                "Coinciden": "Sí" if len(valores_normalizados) == 1 else "No",
                "Valores observados": " | ".join(f"{metodo}: {valor}" for metodo, valor in no_vacios),
            })
    df_consistencia = pd.DataFrame(comparaciones_campo)
    print("Revisión de consistencia entre métodos (campos obtenidos por al menos dos métodos):")
    if not df_consistencia.empty:
        display(df_consistencia)
    else:
        print("Todavía no hay campos extraídos por dos o más métodos para comparar.")
else:
    comparacion = pd.DataFrame()
    df_metricas = pd.DataFrame()
    df_consistencia = pd.DataFrame()
    print("Ninguno de los métodos obtuvo datos. Revisa la conexión e intenta ejecutar de nuevo las celdas anteriores.")

print("\nEstado de las ejecuciones:")
for nombre, (registro, tiempo, error) in resultados_por_metodo.items():
    if registro is not None:
        print(f"- {nombre}: correcto")
    else:
        print(f"- {nombre}: no completado. Motivo: {error or 'sin detalles'}")

print("\nInterpretación automática de esta ejecución:")
if not df_metricas.empty:
    mejor_completitud = df_metricas["Completitud (%)"].max()
    metodos_completitud = df_metricas.loc[df_metricas["Completitud (%)"] == mejor_completitud, "Método"].tolist()
    print(f"- Mayor completitud observada: {mejor_completitud:.2f}% ({', '.join(metodos_completitud)}).")
    df_tiempos = df_metricas.dropna(subset=["Tiempo (s)"])
    if not df_tiempos.empty:
        menor_tiempo = df_tiempos["Tiempo (s)"].min()
        metodos_rapidos = df_tiempos.loc[df_tiempos["Tiempo (s)"] == menor_tiempo, "Método"].tolist()
        print(f"- Menor tiempo medido: {menor_tiempo:.3f} s ({', '.join(metodos_rapidos)}). Los tiempos dependen del entorno y la conexión.")
if len(resultados_validos) == len(resultados_por_metodo):
    if not df_consistencia.empty and (df_consistencia["Coinciden"] == "Sí").all():
        print("- Los campos comparables coinciden entre los tres métodos.")
    elif not df_consistencia.empty:
        diferencias = df_consistencia.loc[df_consistencia["Coinciden"] == "No", "Campo"].tolist()
        print("- Revisa los siguientes campos que no coinciden: " + ", ".join(diferencias) + ".")
else:
    pendientes = [nombre for nombre, (registro, _, _) in resultados_por_metodo.items() if registro is None]
    print("- La comparación aún es parcial; no se debe afirmar que los tres métodos funcionaron. Pendientes: " + ", ".join(pendientes) + ".")

Comparación de los campos extraídos:


,titulo,precio,disponibilidad,upc,tipo_producto,precio_sin_impuestos,precio_con_impuestos,impuesto,numero_resenas,calificacion
Metodo,,,,,,,,,,
BeautifulSoup,A Light in the Attic,£51.77,In stock (22 available),a897fe39b1053632,Books,£51.77,£51.77,£0.00,0,3 de 5
Selenium,A Light in the Attic,£51.77,In stock (22 available),a897fe39b1053632,Books,£51.77,£51.77,£0.00,0,3 de 5
Scrapy,A Light in the Attic,£51.77,In stock (22 available),a897fe39b1053632,Books,£51.77,£51.77,£0.00,0,3 de 5


Métricas por método:


,Método,Campos completos,Campos esperados,Completitud (%),Tiempo (s)
0,BeautifulSoup,10,10,100.0,0.882
1,Selenium,10,10,100.0,6.679
2,Scrapy,10,10,100.0,4.666


Revisión de consistencia entre métodos (campos obtenidos por al menos dos métodos):


,Campo,Métodos con valor,Coinciden,Valores observados
0,titulo,3,Sí,BeautifulSoup: A Light in the Attic | Selenium...
1,precio,3,Sí,BeautifulSoup: £51.77 | Selenium: £51.77 | Scr...
2,disponibilidad,3,Sí,BeautifulSoup: In stock (22 available) | Selen...
3,upc,3,Sí,BeautifulSoup: a897fe39b1053632 | Selenium: a8...
4,tipo_producto,3,Sí,BeautifulSoup: Books | Selenium: Books | Scrap...
5,precio_sin_impuestos,3,Sí,BeautifulSoup: £51.77 | Selenium: £51.77 | Scr...
6,precio_con_impuestos,3,Sí,BeautifulSoup: £51.77 | Selenium: £51.77 | Scr...
7,impuesto,3,Sí,BeautifulSoup: £0.00 | Selenium: £0.00 | Scrap...
8,numero_resenas,3,Sí,BeautifulSoup: 0 | Selenium: 0 | Scrapy: 0
9,calificacion,3,Sí,BeautifulSoup: 3 de 5 | Selenium: 3 de 5 | Scr...



Estado de las ejecuciones:
- BeautifulSoup: correcto
- Selenium: correcto
- Scrapy: correcto

Interpretación automática de esta ejecución:
- Mayor completitud observada: 100.00% (BeautifulSoup, Selenium, Scrapy).
- Menor tiempo medido: 0.882 s (BeautifulSoup). Los tiempos dependen del entorno y la conexión.
- Los campos comparables coinciden entre los tres métodos.


## 7. Exportación de resultados

Esta celda guarda la comparación, las métricas y la revisión de coincidencias en archivos CSV.

In [14]:
carpeta_salida = Path.cwd()

if not comparacion.empty:
    ruta_comparacion = carpeta_salida / "comparacion_scraping_ea1.csv"
    comparacion.to_csv(ruta_comparacion, encoding="utf-8-sig")
    print("Archivo guardado:", ruta_comparacion)

if not df_metricas.empty:
    ruta_metricas = carpeta_salida / "metricas_scraping_ea1.csv"
    df_metricas.to_csv(ruta_metricas, index=False, encoding="utf-8-sig")
    print("Archivo guardado:", ruta_metricas)

if not df_consistencia.empty:
    ruta_consistencia = carpeta_salida / "consistencia_scraping_ea1.csv"
    df_consistencia.to_csv(ruta_consistencia, index=False, encoding="utf-8-sig")
    print("Archivo guardado:", ruta_consistencia)

print("\nEn Google Colab, los archivos quedan en el panel Archivos de la izquierda y se pueden descargar desde allí.")

Archivo guardado: /content/comparacion_scraping_ea1.csv
Archivo guardado: /content/metricas_scraping_ea1.csv
Archivo guardado: /content/consistencia_scraping_ea1.csv

En Google Colab, los archivos quedan en el panel Archivos de la izquierda y se pueden descargar desde allí.
